In [ ]:
# @title 1. Install, imports, Drive mount, device check

%pip install -q -U flax optax tensorflow tensorflow-datasets pandas

import os
import gc
import re
import json
import time
import random as py_random
from pathlib import Path
from typing import Any, Dict
from functools import partial

import numpy as np
import pandas as pd

import jax
import jax.numpy as jnp
from jax import random, lax

import flax
import flax.linen as nn
from flax import serialization, jax_utils, traverse_util
from flax.training import train_state

import optax

import tensorflow as tf
import tensorflow_datasets as tfds

from google.colab import drive
from tqdm.auto import tqdm

tf.get_logger().setLevel("ERROR")
drive.mount("/content/drive", force_remount=True)

NUM_DEVICES = jax.local_device_count()

print("JAX version:", jax.__version__)
print("Backend:", jax.default_backend())
print("Devices:", jax.devices())
print("NUM_DEVICES:", NUM_DEVICES)

if NUM_DEVICES < 1:
    raise RuntimeError("No JAX devices found.")

In [ ]:
# @title 2. All global config in one place

DRIVE_ROOT = Path("/content/drive/MyDrive")

RUN_ROOT = DRIVE_ROOT / "representation_bank"
RUN_GROUP = "repbank_clean_v1"

TFDS_DATA_DIR = DRIVE_ROOT / "tfds_data"
IMAGENET100_ROOT = DRIVE_ROOT / "imagenet100"

RUN_ROOT.mkdir(parents=True, exist_ok=True)
TFDS_DATA_DIR.mkdir(parents=True, exist_ok=True)
# The submitted paper reports CIFAR-10 results; other dataset configs are optional extensions.

DATASET_NAMES = ["cifar10", "cifar100", "imagenet100"]
MODEL_NAMES = ["resnet18", "small_vit"]
SEEDS = list(range(10))

DATASET_CFG = {
    "cifar10": {
        "num_classes": 10,
        "image_size": 32,
        "train_split": "train",
        "eval_split": "test",
        "mean": (0.4914, 0.4822, 0.4465),
        "std":  (0.2470, 0.2435, 0.2616),
    },
    "cifar100": {
        "num_classes": 100,
        "image_size": 32,
        "train_split": "train",
        "eval_split": "test",
        "mean": (0.5071, 0.4867, 0.4408),
        "std":  (0.2675, 0.2565, 0.2761),
    },
    "imagenet100": {
        "num_classes": 100,
        "image_size": 224,
        "train_split": "train",
        "eval_split": "val",
        "mean": (0.485, 0.456, 0.406),
        "std":  (0.229, 0.224, 0.225),
    },
}

TRAINING_CFG = {
    ("cifar10", "resnet18"): dict(
        epochs=200, batch_size=512, optimizer="sgd", lr=0.10,
        weight_decay=5e-4, label_smoothing=0.10, warmup_epochs=5,
        min_epochs=60, patience=35, target_acc=0.94
    ),
    ("cifar100", "resnet18"): dict(
        epochs=240, batch_size=512, optimizer="sgd", lr=0.10,
        weight_decay=5e-4, label_smoothing=0.10, warmup_epochs=5,
        min_epochs=80, patience=40, target_acc=0.75
    ),
    ("imagenet100", "resnet18"): dict(
        epochs=100, batch_size=256, optimizer="sgd", lr=0.10,
        weight_decay=1e-4, label_smoothing=0.10, warmup_epochs=5,
        min_epochs=30, patience=20, target_acc=0.80
    ),
    ("cifar10", "small_vit"): dict(
        epochs=300, batch_size=512, optimizer="adamw", lr=3e-4,
        weight_decay=5e-2, label_smoothing=0.10, warmup_epochs=20,
        min_epochs=100, patience=50, target_acc=0.90
    ),
    ("cifar100", "small_vit"): dict(
        epochs=300, batch_size=512, optimizer="adamw", lr=4e-4,
        weight_decay=5e-2, label_smoothing=0.10, warmup_epochs=20,
        min_epochs=100, patience=50, target_acc=0.68
    ),
    ("imagenet100", "small_vit"): dict(
        epochs=120, batch_size=128, optimizer="adamw", lr=4e-4,
        weight_decay=5e-2, label_smoothing=0.10, warmup_epochs=10,
        min_epochs=35, patience=25, target_acc=0.76
    ),
}

def natural_key(s: str):
    return [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", s)]

def set_all_seeds(seed: int):
    np.random.seed(seed)
    py_random.seed(seed)
    tf.random.set_seed(seed)

def run_dir(save_root, run_group, dataset_name, model_name, seed):
    return Path(save_root) / run_group / dataset_name / model_name / f"seed_{seed:02d}"

def save_json(path: Path, obj: Dict[str, Any]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def load_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)

def shard_array(x: np.ndarray):
    n = x.shape[0]
    if n % NUM_DEVICES != 0:
        raise ValueError(f"Batch size {n} is not divisible by NUM_DEVICES={NUM_DEVICES}")
    return x.reshape((NUM_DEVICES, n // NUM_DEVICES) + x.shape[1:])

def make_batch_dict(batch):
    images, labels = batch
    return {
        "images": shard_array(images),
        "labels": shard_array(labels),
    }

def numpy_prefetch_iter(tf_dataset, prefetch_size=2):
    generator = (make_batch_dict(batch) for batch in tfds.as_numpy(tf_dataset))
    return jax_utils.prefetch_to_device(generator, prefetch_size)

print("RUN_ROOT:", RUN_ROOT)
print("RUN_GROUP:", RUN_GROUP)
print("TFDS_DATA_DIR:", TFDS_DATA_DIR)
print("IMAGENET100_ROOT:", IMAGENET100_ROOT)
print("DATASET_NAMES:", DATASET_NAMES)
print("MODEL_NAMES:", MODEL_NAMES)

In [ ]:
# @title 3. Data pipeline

AUTOTUNE = tf.data.AUTOTUNE

def normalize_image(image, mean, std):
    image = tf.cast(image, tf.float32) / 255.0
    mean = tf.constant(mean, dtype=tf.float32)[None, None, :]
    std = tf.constant(std, dtype=tf.float32)[None, None, :]
    return (image - mean) / std

def resize_shorter_side(image, shorter=256):
    shape = tf.cast(tf.shape(image)[:2], tf.float32)
    scale = shorter / tf.reduce_min(shape)
    new_shape = tf.cast(tf.round(shape * scale), tf.int32)
    return tf.image.resize(image, new_shape, antialias=True)

def preprocess_cifar_train(idx, image, label, dataset_name, seed):
    cfg = DATASET_CFG[dataset_name]
    image = tf.image.resize_with_crop_or_pad(image, 40, 40)
    seed_pair = tf.stack([
        tf.cast(seed, tf.int32),
        tf.cast(idx % 2_000_000_000, tf.int32),
    ])
    image = tf.image.stateless_random_crop(image, size=(32, 32, 3), seed=seed_pair)
    image = tf.image.stateless_random_flip_left_right(
        image, seed=seed_pair + tf.constant([0, 1], dtype=tf.int32)
    )
    image = normalize_image(image, cfg["mean"], cfg["std"])
    return image, tf.cast(label, tf.int32)

def preprocess_cifar_eval(image, label, dataset_name):
    cfg = DATASET_CFG[dataset_name]
    image = normalize_image(image, cfg["mean"], cfg["std"])
    return image, tf.cast(label, tf.int32)

def preprocess_imagenet_train(idx, image, label, seed):
    cfg = DATASET_CFG["imagenet100"]
    image = resize_shorter_side(image, shorter=256)
    seed_pair = tf.stack([
        tf.cast(seed, tf.int32),
        tf.cast(idx % 2_000_000_000, tf.int32),
    ])
    image = tf.image.stateless_random_crop(image, size=(224, 224, 3), seed=seed_pair)
    image = tf.image.stateless_random_flip_left_right(
        image, seed=seed_pair + tf.constant([0, 1], dtype=tf.int32)
    )
    image = normalize_image(image, cfg["mean"], cfg["std"])
    return image, tf.cast(label, tf.int32)

def preprocess_imagenet_eval(image, label):
    cfg = DATASET_CFG["imagenet100"]
    image = resize_shorter_side(image, shorter=256)
    image = tf.image.resize_with_crop_or_pad(image, 224, 224)
    image = normalize_image(image, cfg["mean"], cfg["std"])
    return image, tf.cast(label, tf.int32)

def build_cifar_dataset(dataset_name, split, batch_size, training, seed):
    builder = tfds.builder(dataset_name, data_dir=str(TFDS_DATA_DIR))
    builder.download_and_prepare()

    ds = builder.as_dataset(
        split=split,
        as_supervised=True,
        shuffle_files=training,
        read_config=tfds.ReadConfig(shuffle_seed=seed),
    )

    if training:
        ds = ds.cache()
        ds = ds.shuffle(50_000, seed=seed, reshuffle_each_iteration=True)
        ds = ds.repeat()
        ds = ds.enumerate()
        ds = ds.map(
            lambda i, xy: preprocess_cifar_train(i, xy[0], xy[1], dataset_name, seed),
            num_parallel_calls=AUTOTUNE,
        )
    else:
        ds = ds.map(
            lambda x, y: preprocess_cifar_eval(x, y, dataset_name),
            num_parallel_calls=AUTOTUNE,
        )

    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(AUTOTUNE)

    num_examples = builder.info.splits[split].num_examples
    class_names = builder.info.features["label"].names
    return ds, num_examples, class_names

def build_imagenet100_dataset(split, batch_size, training, seed):
    builder = tfds.ImageFolder(str(IMAGENET100_ROOT))

    ds = builder.as_dataset(
        split=split,
        as_supervised=True,
        shuffle_files=training,
        read_config=tfds.ReadConfig(shuffle_seed=seed),
    )

    if training:
        ds = ds.shuffle(100_000, seed=seed, reshuffle_each_iteration=True)
        ds = ds.repeat()
        ds = ds.enumerate()
        ds = ds.map(
            lambda i, xy: preprocess_imagenet_train(i, xy[0], xy[1], seed),
            num_parallel_calls=AUTOTUNE,
        )
    else:
        ds = ds.map(preprocess_imagenet_eval, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(AUTOTUNE)

    num_examples = builder.info.splits[split].num_examples
    class_names = builder.info.features["label"].names
    return ds, num_examples, class_names

def make_dataloaders(dataset_name, batch_size, seed):
    if batch_size % NUM_DEVICES != 0:
        raise ValueError(f"batch_size={batch_size} must be divisible by NUM_DEVICES={NUM_DEVICES}")

    if dataset_name in ("cifar10", "cifar100"):
        train_ds, train_size, class_names = build_cifar_dataset(
            dataset_name,
            DATASET_CFG[dataset_name]["train_split"],
            batch_size,
            True,
            seed,
        )
        eval_ds, eval_size, _ = build_cifar_dataset(
            dataset_name,
            DATASET_CFG[dataset_name]["eval_split"],
            batch_size,
            False,
            seed,
        )
    elif dataset_name == "imagenet100":
        train_ds, train_size, class_names = build_imagenet100_dataset(
            DATASET_CFG[dataset_name]["train_split"],
            batch_size,
            True,
            seed,
        )
        eval_ds, eval_size, _ = build_imagenet100_dataset(
            DATASET_CFG[dataset_name]["eval_split"],
            batch_size,
            False,
            seed,
        )
    else:
        raise ValueError(dataset_name)

    steps_per_epoch = train_size // batch_size
    eval_steps = eval_size // batch_size
    train_iter = iter(numpy_prefetch_iter(train_ds, prefetch_size=2))
    return train_iter, eval_ds, steps_per_epoch, eval_steps, class_names

print("Data pipeline ready.")

In [ ]:
# @title 4. Models: ResNet-18 and small ViT, both with intermediate activations (fixed names)

class TrainState(train_state.TrainState):
    batch_stats: Any = flax.struct.field(pytree_node=True, default_factory=dict)

class ResidualBlock(nn.Module):
    features: int
    stride: int = 1

    @nn.compact
    def __call__(self, x, train: bool):
        residual = x

        y = nn.Conv(
            self.features, (3, 3), strides=(self.stride, self.stride),
            padding="SAME", use_bias=False
        )(x)
        y = nn.BatchNorm(
            use_running_average=not train, momentum=0.9, epsilon=1e-5
        )(y)
        y = nn.relu(y)

        y = nn.Conv(
            self.features, (3, 3), strides=(1, 1),
            padding="SAME", use_bias=False
        )(y)
        y = nn.BatchNorm(
            use_running_average=not train,
            momentum=0.9,
            epsilon=1e-5,
            scale_init=nn.initializers.zeros,
        )(y)

        if residual.shape != y.shape:
            residual = nn.Conv(
                self.features, (1, 1), strides=(self.stride, self.stride),
                use_bias=False
            )(residual)
            residual = nn.BatchNorm(
                use_running_average=not train, momentum=0.9, epsilon=1e-5
            )(residual)

        out = nn.relu(residual + y)
        return out

class ResNet18(nn.Module):
    num_classes: int
    image_size: int

    @nn.compact
    def __call__(self, x, train: bool):
        small_input = self.image_size <= 64

        if small_input:
            x = nn.Conv(
                64, (3, 3), strides=(1, 1), padding="SAME",
                use_bias=False, name="stem_conv"
            )(x)
        else:
            x = nn.Conv(
                64, (7, 7), strides=(2, 2), padding="SAME",
                use_bias=False, name="stem_conv"
            )(x)

        x = nn.BatchNorm(
            use_running_average=not train, momentum=0.9, epsilon=1e-5, name="stem_bn"
        )(x)
        x = nn.relu(x)
        self.sow("intermediates", "act_stem", x)

        if not small_input:
            x = nn.max_pool(x, window_shape=(3, 3), strides=(2, 2), padding="SAME")

        block_specs = [
            (64, 1), (64, 1),
            (128, 2), (128, 1),
            (256, 2), (256, 1),
            (512, 2), (512, 1),
        ]

        for i, (features, stride) in enumerate(block_specs, start=1):
            x = ResidualBlock(features=features, stride=stride, name=f"block{i}")(x, train=train)
            self.sow("intermediates", f"act_block{i}", x)

        x = jnp.mean(x, axis=(1, 2))
        self.sow("intermediates", "act_pre_logits", x)
        logits = nn.Dense(self.num_classes, name="head")(x)
        return logits

class DropPath(nn.Module):
    rate: float = 0.0

    @nn.compact
    def __call__(self, x, train: bool):
        if (not train) or self.rate == 0.0:
            return x
        keep_prob = 1.0 - self.rate
        mask_shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = jax.random.bernoulli(self.make_rng("drop_path"), p=keep_prob, shape=mask_shape)
        return x * mask.astype(x.dtype) / keep_prob

class MLP(nn.Module):
    hidden_dim: int
    out_dim: int
    drop_rate: float = 0.0

    @nn.compact
    def __call__(self, x, train: bool):
        x = nn.Dense(self.hidden_dim)(x)
        x = nn.gelu(x)
        x = nn.Dropout(self.drop_rate)(x, deterministic=not train)
        x = nn.Dense(self.out_dim)(x)
        x = nn.Dropout(self.drop_rate)(x, deterministic=not train)
        return x

class EncoderBlock(nn.Module):
    dim: int
    num_heads: int
    mlp_ratio: float = 4.0
    drop_rate: float = 0.0
    drop_path_rate: float = 0.0

    @nn.compact
    def __call__(self, x, train: bool):
        y = nn.LayerNorm()(x)
        y = nn.MultiHeadDotProductAttention(
            num_heads=self.num_heads,
            qkv_features=self.dim,
            out_features=self.dim,
            dropout_rate=self.drop_rate,
        )(y, y, deterministic=not train)
        y = DropPath(self.drop_path_rate)(y, train=train)
        x = x + y

        y = nn.LayerNorm()(x)
        y = MLP(
            hidden_dim=int(self.dim * self.mlp_ratio),
            out_dim=self.dim,
            drop_rate=self.drop_rate,
        )(y, train=train)
        y = DropPath(self.drop_path_rate)(y, train=train)
        x = x + y
        return x

class SmallViT(nn.Module):
    num_classes: int
    image_size: int
    patch_size: int
    embed_dim: int = 256
    depth: int = 8
    num_heads: int = 8
    mlp_ratio: float = 4.0
    drop_rate: float = 0.0
    drop_path_rate: float = 0.10

    @nn.compact
    def __call__(self, x, train: bool):
        x = nn.Conv(
            features=self.embed_dim,
            kernel_size=(self.patch_size, self.patch_size),
            strides=(self.patch_size, self.patch_size),
            padding="VALID",
            name="patch_embed",
        )(x)
        self.sow("intermediates", "act_patch_grid", x)

        b, h, w, c = x.shape
        x = x.reshape((b, h * w, c))
        self.sow("intermediates", "act_tokens_in", x)

        cls_token = self.param("cls_token", nn.initializers.zeros, (1, 1, c))
        pos_embed = self.param(
            "pos_embedding",
            nn.initializers.normal(stddev=0.02),
            (1, x.shape[1] + 1, c),
        )

        cls_tokens = jnp.tile(cls_token, (b, 1, 1))
        x = jnp.concatenate([cls_tokens, x], axis=1)
        x = x + pos_embed
        x = nn.Dropout(self.drop_rate)(x, deterministic=not train)

        for i in range(self.depth):
            dpr = self.drop_path_rate * (i / max(1, self.depth - 1))
            x = EncoderBlock(
                dim=self.embed_dim,
                num_heads=self.num_heads,
                mlp_ratio=self.mlp_ratio,
                drop_rate=self.drop_rate,
                drop_path_rate=dpr,
                name=f"encoderblock_{i+1}",
            )(x, train=train)
            self.sow("intermediates", f"act_encoderblock_{i+1}", x)

        x = nn.LayerNorm(name="encoder_norm")(x)
        self.sow("intermediates", "act_pre_logits", x[:, 0])
        logits = nn.Dense(self.num_classes, name="head")(x[:, 0])
        return logits

def build_model(model_name: str, dataset_name: str):
    num_classes = DATASET_CFG[dataset_name]["num_classes"]
    image_size = DATASET_CFG[dataset_name]["image_size"]

    if model_name == "resnet18":
        model = ResNet18(num_classes=num_classes, image_size=image_size)
        model_cfg = {
            "model_name": model_name,
            "num_classes": num_classes,
            "image_size": image_size,
        }
        return model, model_cfg

    if model_name == "small_vit":
        patch_size = 4 if image_size <= 32 else 16
        embed_dim = 256 if image_size <= 32 else 384
        num_heads = 8 if image_size <= 32 else 6

        model = SmallViT(
            num_classes=num_classes,
            image_size=image_size,
            patch_size=patch_size,
            embed_dim=embed_dim,
            depth=8,
            num_heads=num_heads,
            mlp_ratio=4.0,
            drop_rate=0.0,
            drop_path_rate=0.10,
        )
        model_cfg = {
            "model_name": model_name,
            "num_classes": num_classes,
            "image_size": image_size,
            "patch_size": patch_size,
            "embed_dim": embed_dim,
            "depth": 8,
            "num_heads": num_heads,
            "mlp_ratio": 4.0,
            "drop_rate": 0.0,
            "drop_path_rate": 0.10,
        }
        return model, model_cfg

    raise ValueError(model_name)

def get_layer_names(model, dataset_name):
    image_size = DATASET_CFG[dataset_name]["image_size"]
    dummy = jnp.zeros((1, image_size, image_size, 3), dtype=jnp.float32)
    key = random.PRNGKey(0)
    vars0 = model.init(
        {"params": key, "dropout": key, "drop_path": key},
        dummy,
        train=False,
    )
    _, mut = model.apply(vars0, dummy, train=False, mutable=["intermediates"])
    layer_names = sorted(mut["intermediates"].keys(), key=natural_key)
    return list(layer_names)

print("Models ready.")

In [ ]:
# @title 5. Optimizer, schedules, train/eval steps, checkpoint helpers

def decay_mask_from_params(params):
    flat = traverse_util.flatten_dict(params)
    mask = {}
    for key_tuple, value in flat.items():
        key = "/".join(key_tuple)
        use_decay = getattr(value, "ndim", 0) > 1
        if any(tok in key for tok in ["bias", "scale", "BatchNorm", "batchnorm", "pos_embedding", "cls_token"]):
            use_decay = False
        mask[key_tuple] = use_decay
    return traverse_util.unflatten_dict(mask)

def make_lr_schedule(total_steps, base_lr, warmup_steps):
    return optax.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=base_lr,
        warmup_steps=max(1, warmup_steps),
        decay_steps=max(1, total_steps),
        end_value=base_lr * 1e-2,
    )

def init_state_for_run(model, dataset_name, model_name, seed, steps_per_epoch):
    sched = TRAINING_CFG[(dataset_name, model_name)]
    image_size = DATASET_CFG[dataset_name]["image_size"]

    total_steps = sched["epochs"] * steps_per_epoch
    warmup_steps = sched["warmup_epochs"] * steps_per_epoch
    lr_fn = make_lr_schedule(total_steps, sched["lr"], warmup_steps)

    rng = random.PRNGKey(seed)
    params_key, dropout_key, droppath_key = random.split(rng, 3)

    dummy = jnp.zeros((1, image_size, image_size, 3), dtype=jnp.float32)
    variables = model.init(
        {"params": params_key, "dropout": dropout_key, "drop_path": droppath_key},
        dummy,
        train=True,
    )

    params = variables["params"]
    batch_stats = variables.get("batch_stats", {})
    has_batch_stats = bool(batch_stats)

    mask = decay_mask_from_params(params)

    if sched["optimizer"] == "sgd":
        tx = optax.chain(
            optax.add_decayed_weights(sched["weight_decay"], mask=mask),
            optax.sgd(learning_rate=lr_fn, momentum=0.9, nesterov=True),
        )
    elif sched["optimizer"] == "adamw":
        tx = optax.chain(
            optax.clip_by_global_norm(1.0),
            optax.adamw(
                learning_rate=lr_fn,
                weight_decay=sched["weight_decay"],
                b1=0.9,
                b2=0.999,
                mask=mask,
            ),
        )
    else:
        raise ValueError(sched["optimizer"])

    state = TrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx,
        batch_stats=batch_stats,
    )
    return state, lr_fn, has_batch_stats

def smooth_one_hot(labels, num_classes, smoothing):
    off = smoothing / num_classes
    on = 1.0 - smoothing + off
    y = jax.nn.one_hot(labels, num_classes)
    return y * (on - off) + off

def make_train_step(num_classes, label_smoothing, has_batch_stats):
    @partial(jax.pmap, axis_name="batch", donate_argnums=(0,))
    def train_step(state, batch, rng):
        def loss_fn(params):
            variables = {"params": params}
            if has_batch_stats:
                variables["batch_stats"] = state.batch_stats
                logits, new_model_state = state.apply_fn(
                    variables,
                    batch["images"],
                    train=True,
                    rngs={"dropout": rng, "drop_path": rng},
                    mutable=["batch_stats"],
                )
            else:
                logits = state.apply_fn(
                    variables,
                    batch["images"],
                    train=True,
                    rngs={"dropout": rng, "drop_path": rng},
                )
                new_model_state = {}

            targets = smooth_one_hot(batch["labels"], num_classes, label_smoothing)
            loss = optax.softmax_cross_entropy(logits=logits, labels=targets).mean()
            return loss, (logits, new_model_state)

        (loss, (logits, new_model_state)), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)

        grads = lax.pmean(grads, axis_name="batch")
        loss = lax.pmean(loss, axis_name="batch")
        acc = lax.pmean(
            jnp.mean(jnp.argmax(logits, axis=-1) == batch["labels"]),
            axis_name="batch",
        )

        if has_batch_stats:
            new_batch_stats = lax.pmean(new_model_state["batch_stats"], axis_name="batch")
        else:
            new_batch_stats = state.batch_stats

        new_state = state.apply_gradients(grads=grads, batch_stats=new_batch_stats)
        metrics = {"loss": loss, "acc": acc}
        return new_state, metrics

    return train_step

def make_eval_step(num_classes, has_batch_stats):
    @partial(jax.pmap, axis_name="batch")
    def eval_step(state, batch):
        variables = {"params": state.params}
        if has_batch_stats:
            variables["batch_stats"] = state.batch_stats

        logits = state.apply_fn(variables, batch["images"], train=False)
        targets = jax.nn.one_hot(batch["labels"], num_classes)

        loss = optax.softmax_cross_entropy(logits=logits, labels=targets).mean()
        loss = lax.pmean(loss, axis_name="batch")
        acc = lax.pmean(
            jnp.mean(jnp.argmax(logits, axis=-1) == batch["labels"]),
            axis_name="batch",
        )
        return {"loss": loss, "acc": acc}

    return eval_step

def scalarize_metrics(metrics_list):
    out = {}
    for key in metrics_list[0].keys():
        out[key] = float(np.mean([float(m[key]) for m in metrics_list]))
    return out

def unreplicate_state(pstate):
    return jax_utils.unreplicate(pstate)

def save_train_state(path: Path, state: TrainState):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        f.write(serialization.to_bytes(state))

def restore_train_state(path: Path, template_state: TrainState):
    with open(path, "rb") as f:
        return serialization.from_bytes(template_state, f.read())

def save_analysis_payload(path: Path, state: TrainState):
    payload = {
        "params": state.params,
        "batch_stats": state.batch_stats,
        "step": np.asarray(int(state.step), dtype=np.int32),
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        f.write(serialization.to_bytes(payload))

print("Training utilities ready.")

In [ ]:
# @title 6. Single-run trainer with resume, best checkpoint, and early stopping

def train_one_run(dataset_name: str, model_name: str, seed: int,
                  save_root=None, run_group=None, verbose=True):
    if save_root is None:
        save_root = RUN_ROOT
    if run_group is None:
        run_group = RUN_GROUP

    set_all_seeds(seed)

    sched = TRAINING_CFG[(dataset_name, model_name)]
    cfg = DATASET_CFG[dataset_name]
    rdir = run_dir(save_root, run_group, dataset_name, model_name, seed)
    rdir.mkdir(parents=True, exist_ok=True)

    done_path = rdir / "done.json"
    if done_path.exists():
        if verbose:
            print("Already finished:", rdir)
        return load_json(done_path)

    if verbose:
        print(f"\n=== {dataset_name} | {model_name} | seed={seed} ===")
        print("Run dir:", rdir)

    train_iter, eval_ds, steps_per_epoch, eval_steps, class_names = make_dataloaders(
        dataset_name, batch_size=sched["batch_size"], seed=seed
    )

    model, model_cfg = build_model(model_name, dataset_name)
    layer_names = get_layer_names(model, dataset_name)

    state, lr_fn, has_batch_stats = init_state_for_run(
        model, dataset_name, model_name, seed, steps_per_epoch
    )

    meta = {
        "dataset_name": dataset_name,
        "model_name": model_name,
        "seed": seed,
        "dataset_cfg": cfg,
        "model_cfg": model_cfg,
        "schedule": sched,
        "layer_names": layer_names,
        "class_names": class_names,
        "created_at_unix": time.time(),
    }
    meta_path = rdir / "meta.json"
    if not meta_path.exists():
        save_json(meta_path, meta)

    full_state_path = rdir / "last_train_state.msgpack"
    run_state_path = rdir / "run_state.json"
    history_path = rdir / "history.csv"

    start_epoch = 1
    best_eval_acc = -1.0
    best_epoch = -1
    history = []

    if full_state_path.exists() and run_state_path.exists():
        if verbose:
            print("Resuming from:", full_state_path)
        state = restore_train_state(full_state_path, state)
        rs = load_json(run_state_path)
        start_epoch = int(rs["epoch_completed"]) + 1
        best_eval_acc = float(rs["best_eval_acc"])
        best_epoch = int(rs["best_epoch"])
        if history_path.exists():
            history = pd.read_csv(history_path).to_dict("records")

    pstate = jax_utils.replicate(state)
    p_train_step = make_train_step(
        num_classes=cfg["num_classes"],
        label_smoothing=sched["label_smoothing"],
        has_batch_stats=has_batch_stats,
    )
    p_eval_step = make_eval_step(
        num_classes=cfg["num_classes"],
        has_batch_stats=has_batch_stats,
    )

    master_rng = random.PRNGKey(seed + 12345)

    for epoch in range(start_epoch, sched["epochs"] + 1):
        t0 = time.time()

        train_metrics = []
        for _ in tqdm(range(steps_per_epoch), disable=not verbose, leave=False, desc=f"train e{epoch:03d}"):
            batch = next(train_iter)
            master_rng, step_rng = random.split(master_rng)
            step_rngs = random.split(step_rng, NUM_DEVICES)
            pstate, metrics = p_train_step(pstate, batch, step_rngs)
            metrics = jax.device_get(jax_utils.unreplicate(metrics))
            train_metrics.append(metrics)

        train_log = scalarize_metrics(train_metrics)

        eval_metrics = []
        eval_iter = numpy_prefetch_iter(eval_ds, prefetch_size=2)
        for _ in tqdm(range(eval_steps), disable=not verbose, leave=False, desc=f"eval  e{epoch:03d}"):
            batch = next(eval_iter)
            metrics = p_eval_step(pstate, batch)
            metrics = jax.device_get(jax_utils.unreplicate(metrics))
            eval_metrics.append(metrics)

        eval_log = scalarize_metrics(eval_metrics)
        host_state = unreplicate_state(pstate)

        save_train_state(rdir / "last_train_state.msgpack", host_state)
        save_analysis_payload(rdir / "last_analysis.msgpack", host_state)

        improved = eval_log["acc"] > best_eval_acc
        if improved:
            best_eval_acc = eval_log["acc"]
            best_epoch = epoch
            save_analysis_payload(rdir / "best_analysis.msgpack", host_state)
            save_json(
                rdir / "best_metrics.json",
                {
                    "best_eval_acc": best_eval_acc,
                    "best_epoch": best_epoch,
                    "step": int(host_state.step),
                },
            )

        row = {
            "epoch": epoch,
            "step": int(host_state.step),
            "lr": float(lr_fn(int(host_state.step))),
            "train_loss": train_log["loss"],
            "train_acc": train_log["acc"],
            "eval_loss": eval_log["loss"],
            "eval_acc": eval_log["acc"],
            "best_eval_acc_so_far": best_eval_acc,
            "epoch_seconds": time.time() - t0,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)

        save_json(
            run_state_path,
            {
                "epoch_completed": epoch,
                "best_eval_acc": best_eval_acc,
                "best_epoch": best_epoch,
                "last_step": int(host_state.step),
            },
        )

        if verbose:
            print(
                f"[{dataset_name} | {model_name} | seed={seed:02d}] "
                f"epoch {epoch:03d}/{sched['epochs']}  "
                f"train_acc={row['train_acc']:.4f}  "
                f"eval_acc={row['eval_acc']:.4f}  "
                f"best={best_eval_acc:.4f}  "
                f"lr={row['lr']:.6f}"
            )

        reached_target = (epoch >= sched["min_epochs"]) and (best_eval_acc >= sched["target_acc"])
        patience_exhausted = (epoch - best_epoch) >= sched["patience"]

        if reached_target:
            if verbose:
                print(f"Stopping early: reached target_acc={sched['target_acc']:.4f}")
            break

        if patience_exhausted:
            if verbose:
                print(f"Stopping early: patience exhausted ({sched['patience']} epochs without improvement)")
            break

        gc.collect()

    result = {
        "run_dir": str(rdir),
        "dataset_name": dataset_name,
        "model_name": model_name,
        "seed": seed,
        "best_eval_acc": best_eval_acc,
        "best_epoch": best_epoch,
        "history_path": str(history_path),
    }
    save_json(done_path, result)

    if verbose:
        print("Finished:", rdir)
        print("Best eval acc:", best_eval_acc, "at epoch", best_epoch)

    return result

print("Single-run trainer ready.")

In [ ]:
# @title 7. Optional smoke test: run one model before launching the full bank

# Uncomment this first to make sure everything compiles and trains correctly.
result = train_one_run("cifar10", "resnet18", 0, verbose=True)
print(result)

In [ ]:
# @title 8. Full launcher for the 60-model bank, with auto-resume and subset controls

ONLY_DATASETS = None          # e.g. ["cifar10"]
ONLY_DATASETS = ["cifar10", "cifar100"]
ONLY_MODELS = None            # e.g. ["resnet18"]
ONLY_SEEDS = None             # e.g. [0, 1, 2]
MAX_RUNS_THIS_SESSION = None  # e.g. 2

def run_is_finished(dataset_name, model_name, seed, save_root=None, run_group=None):
    if save_root is None:
        save_root = RUN_ROOT
    if run_group is None:
        run_group = RUN_GROUP
    rdir = run_dir(save_root, run_group, dataset_name, model_name, seed)
    return (rdir / "done.json").exists()

grid = []
for dataset_name in DATASET_NAMES:
    if ONLY_DATASETS is not None and dataset_name not in ONLY_DATASETS:
        continue
    for model_name in MODEL_NAMES:
        if ONLY_MODELS is not None and model_name not in ONLY_MODELS:
            continue
        for seed in SEEDS:
            if ONLY_SEEDS is not None and seed not in ONLY_SEEDS:
                continue
            grid.append((dataset_name, model_name, seed))

print("Planned runs:", len(grid))
for item in grid[:10]:
    print(" ", item)
if len(grid) > 10:
    print(" ...")

completed = 0
launched = 0

for dataset_name, model_name, seed in grid:
    if run_is_finished(dataset_name, model_name, seed):
        print(f"SKIP done: {dataset_name} | {model_name} | seed={seed}")
        completed += 1
        continue

    train_one_run(
        dataset_name=dataset_name,
        model_name=model_name,
        seed=seed,
        save_root=RUN_ROOT,
        run_group=RUN_GROUP,
        verbose=True,
    )
    launched += 1

    if MAX_RUNS_THIS_SESSION is not None and launched >= MAX_RUNS_THIS_SESSION:
        print(f"Stopping because MAX_RUNS_THIS_SESSION={MAX_RUNS_THIS_SESSION}")
        break

print("Completed already:", completed)
print("Launched now:", launched)